# 01 · Build the *same* agent three ways
### No framework → Bedrock Agents (Agent Builder) → Strands

One task. Three philosophies. Feeling the difference is the fastest way to understand *why* AgentCore exists and *where* Strands fits.

**The task:** a tiny TravelMind skill — *"tell the traveler a flight's status, and if it's delayed, by how much."* One tool, `get_flight_status`. We'll give the same tool to:

1. **No framework** — a raw `bedrock-runtime` Converse **tool-use loop**. You write every step.
2. **Bedrock Agents / Agent Builder** — config-first. **AWS runs the loop**; you supply the tool result.
3. **Strands** — code-first. A few lines; the framework runs the loop for you.

```mermaid
flowchart TB
    T[Task: get flight status] --> W1[Way 1: you write the loop<br/>total control, most code]
    T --> W2[Way 2: AWS runs the loop<br/>config-first, least code to a managed agent]
    T --> W3[Way 3: Strands runs the loop<br/>code-first, the sweet spot]
    W3 --> R[AgentCore Runtime hosts this happily -> notebook 02]
```

> Prereqs: run **`00_setup_and_sanity.ipynb`** first. You need model access in `us-east-1`. The tool here uses **mock flight data** so the logic runs without any external API — the only thing hitting AWS is the model call.


## Shared setup — one tool, one source of truth

We define the flight data and a plain lookup helper **once**. All three "ways" call this same `lookup_flight`, so the only thing that changes between them is *how the agent loop is wired*, not the business logic.


In [ ]:
import os, json, uuid, boto3
from botocore.exceptions import ClientError

os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
REGION       = "us-east-1"
MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_SONNET = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# Mock "flight database" — stands in for a real airline API (we wire a real one in notebook 04).
FLIGHTS = {
    "BA117": {"flight": "BA117", "status": "Delayed", "from": "LHR", "to": "JFK", "delay_min": 75},
    "AA100": {"flight": "AA100", "status": "On time", "from": "JFK", "to": "LHR", "delay_min": 0},
    "AI191": {"flight": "AI191", "status": "Cancelled", "from": "DEL", "to": "SFO", "delay_min": None},
}

def lookup_flight(flight_number: str) -> dict:
    return FLIGHTS.get(flight_number.upper().replace(" ", ""), {"status": "Unknown flight"})

print(lookup_flight("ba117"))

---
## Way 1 — No framework: the raw Converse tool-use loop

This is what *every* agent framework automates. Understanding it once means you'll never be confused by "magic" again.

**The protocol (Bedrock Converse with tools):**
- You send `messages` plus a `toolConfig` that *describes* your tools (name, description, JSON input schema).
- The model replies. If it wants a tool, the response has `stopReason == "tool_use"` and a `toolUse` block: `{toolUseId, name, input}`.
- **You** run the matching Python function, then send the answer back as a `toolResult` block carrying the **same `toolUseId`** (that id is how the model correlates request↔result).
- Repeat until `stopReason != "tool_use"` — then the model's text is the final answer.

```mermaid
sequenceDiagram
    participant You as Your loop
    participant M as Claude (Converse)
    participant F as get_flight_status()
    You->>M: messages + toolConfig
    M-->>You: stopReason=tool_use, toolUse{id, input}
    You->>F: lookup_flight(input)
    F-->>You: {status, delay_min, ...}
    You->>M: toolResult{same id, json}
    M-->>You: final text
```


In [ ]:
# 1) Describe the tool to the model (Converse toolConfig schema)
TOOL_CONFIG = {
    "tools": [{
        "toolSpec": {
            "name": "get_flight_status",
            "description": "Look up the current status of a flight by its flight number.",
            "inputSchema": {"json": {
                "type": "object",
                "properties": {
                    "flight_number": {"type": "string", "description": "Flight number, e.g. BA117"}
                },
                "required": ["flight_number"],
            }},
        }
    }]
}

# 2) The loop you'd otherwise pay a framework to write for you
def run_no_framework(user_text: str, model: str = MODEL_HAIKU, max_turns: int = 5) -> str:
    rt = boto3.client("bedrock-runtime", region_name=REGION)
    messages = [{"role": "user", "content": [{"text": user_text}]}]

    for _ in range(max_turns):
        resp = rt.converse(
            modelId=model,
            messages=messages,
            toolConfig=TOOL_CONFIG,
            inferenceConfig={"maxTokens": 512},
        )
        assistant_msg = resp["output"]["message"]
        messages.append(assistant_msg)                      # keep the turn in history

        if resp["stopReason"] != "tool_use":                # model is done -> return its text
            return "".join(b.get("text", "") for b in assistant_msg["content"]).strip()

        # model requested one or more tools -> run them, send results back
        tool_results = []
        for block in assistant_msg["content"]:
            if "toolUse" in block:
                tu = block["toolUse"]                        # {toolUseId, name, input}
                result = lookup_flight(**tu["input"])
                tool_results.append({"toolResult": {
                    "toolUseId": tu["toolUseId"],            # MUST echo the same id
                    "content": [{"json": result}],
                }})
        messages.append({"role": "user", "content": tool_results})

    return "Stopped: too many tool turns."

print(run_no_framework("Is BA117 on time? If delayed, by how much?"))

**What you just had to own:** the message history, the tool schema, the `tool_use` detection, the id correlation, the loop guard, and (in real life) retries, timeouts, and parallel tool calls. Total control — and total maintenance. That's the trade.


---
## Way 2 — Bedrock Agents / Agent Builder: let AWS run the loop

You already know this layer. Two faces of it:

### 2a) The console — Agent Builder (the throwback)
In the Bedrock console you create an agent by *describing* it, not coding the loop:
- **Agent** → name + instruction + foundation model (pick Claude 4.x).
- **Action group** → the agent's tools, backed by a **Lambda** (or *RETURN_CONTROL* to run them in your own code).
- **Prepare** the agent → create an **alias** → **Test** in the chat panel.

> 📸 *Capture your own screenshots here from the Bedrock console (us-east-1). We don't ship fabricated console images — yours will match your account exactly.*

The console is great for a quick managed agent. The catch: the loop, prompt, and orchestration are AWS's; you get less control and the agent's tools usually need a deployed Lambda.

### 2b) The same thing in code — *without* a Lambda
`InvokeInlineAgent` defines an agent **at call time** (no `CreateAgent`, no alias, no resources to clean up). We set the tool executor to **`RETURN_CONTROL`**, which means: when the model wants the tool, AWS *hands control back to us* to run it and return the result. Same managed-loop behavior, but the tool runs in this notebook.

```mermaid
sequenceDiagram
    participant You as Your code
    participant A as AgentCore-managed agent loop
    participant F as lookup_flight()
    You->>A: invoke_inline_agent(instruction, tool schema, inputText)
    A-->>You: RETURN_CONTROL {invocationId, function, parameters}
    You->>F: run the function
    F-->>You: result
    You->>A: invoke again with returnControlInvocationResults
    A-->>You: chunk(s) = final text
```


In [ ]:
def _drain(resp):
    """Read one inline-agent event stream: collect text chunks + any RETURN_CONTROL request."""
    text, invocation_id, fn_call = "", None, None
    for event in resp["completion"]:                 # 'completion' is an event stream
        if "chunk" in event:
            text += event["chunk"]["bytes"].decode("utf-8")
        elif "returnControl" in event:
            rc = event["returnControl"]
            invocation_id = rc["invocationId"]
            fn_call = rc["invocationInputs"][0]["functionInvocationInput"]
    return text, invocation_id, fn_call

INSTRUCTION = "You are a flight status assistant. Use the tool to look up status, then answer the traveler plainly."

ACTION_GROUPS = [{
    "actionGroupName": "flight_tools",
    "actionGroupExecutor": {"customControl": "RETURN_CONTROL"},   # run tools in OUR code, no Lambda
    "functionSchema": {"functions": [{
        "name": "get_flight_status",
        "description": "Look up the current status of a flight by its flight number.",
        "parameters": {
            "flight_number": {"type": "string", "description": "e.g. BA117", "required": True}
        },
    }]},
}]

def run_inline_agent(user_text: str, model: str = MODEL_HAIKU) -> str:
    bar = boto3.client("bedrock-agent-runtime", region_name=REGION)
    session_id = "demo-" + uuid.uuid4().hex

    resp = bar.invoke_inline_agent(
        foundationModel=model,
        instruction=INSTRUCTION,
        actionGroups=ACTION_GROUPS,
        sessionId=session_id,
        inputText=user_text,
    )
    text, invocation_id, fn_call = _drain(resp)

    # If the agent handed control back, run the tool and return the result, then continue.
    while fn_call:
        args = {p["name"]: p["value"] for p in fn_call["parameters"]}
        result = lookup_flight(**args)
        resp = bar.invoke_inline_agent(
            foundationModel=model,
            instruction=INSTRUCTION,
            actionGroups=ACTION_GROUPS,
            sessionId=session_id,                 # same session ties the turns together
            inlineSessionState={
                "invocationId": invocation_id,
                "returnControlInvocationResults": [{"functionResult": {
                    "actionGroup": fn_call["actionGroup"],
                    "function": fn_call["function"],
                    "responseBody": {"TEXT": {"body": json.dumps(result)}},
                }}],
            },
        )
        text, invocation_id, fn_call = _drain(resp)
    return text.strip()

print(run_inline_agent("Is BA117 on time? If delayed, by how much?"))

**What changed vs Way 1:** you no longer write the loop, parse `tool_use`, or manage message history — AWS does. You *do* still run the tool (because of `RETURN_CONTROL`) and you parse an event stream. Less control, less code-to-a-working-agent.

> **When you'd use the full `CreateAgent` path instead:** when you want a *persistent, named, versioned* agent with an alias, a Lambda-backed action group, and console testing — i.e. an agent as a managed AWS resource. That path is `CreateAgent` → `CreateAgentActionGroup` (Lambda ARN) → `PrepareAgent` → `CreateAgentAlias` → `InvokeAgent`. It provisions real resources, so we keep it out of an auto-running cell; the inline version above teaches the same loop mechanics without the cleanup.


---
## Way 3 — Strands: code-first, the sweet spot

Now the same agent in the framework AgentCore is happiest hosting.

**Two ideas to internalize:**
- `@tool` turns a plain Python function into a tool. **The function's docstring becomes the tool description** the model reads, and the type hints become the input schema. So a good docstring = a well-described tool.
- `Agent(model=..., tools=[...], system_prompt=...)` wires it up. Calling `agent("...")` runs the *same* Converse tool-use loop you hand-wrote in Way 1 — detection, id correlation, result feedback, retries — all inside the framework.


In [ ]:
from strands import Agent, tool

@tool
def get_flight_status(flight_number: str) -> dict:
    """Look up the current status of a flight by its flight number (e.g. BA117).

    Args:
        flight_number: The flight's number/code, like "BA117" or "AA100".
    """
    return lookup_flight(flight_number)

flight_agent = Agent(
    model=MODEL_HAIKU,
    tools=[get_flight_status],
    system_prompt="You are a flight status assistant. Use the tool, then answer the traveler plainly.",
)

result = flight_agent("Is BA117 on time? If delayed, by how much?")
print(str(result))          # str(result) -> the final text

**Inspecting what happened.** A Strands call returns an `AgentResult` with more than just text — useful for debugging and cost tracking.


In [ ]:
print("stop reason :", result.stop_reason)          # e.g. 'end_turn'
print("token usage :", result.metrics.accumulated_usage)   # input/output tokens
# result.message holds the structured final message (role + content blocks)

That's roughly **8 lines** for what took ~30 in Way 1, with retries and parsing handled for you. This is the object we'll deploy to AgentCore Runtime next.


---
## Side by side

| | Way 1 — No framework | Way 2 — Bedrock Agents / Agent Builder | Way 3 — Strands |
|---|---|---|---|
| Who runs the loop | **You** | **AWS** | **Strands** |
| Lines to a working agent | Most | Few (console) / medium (inline) | Fewest in code |
| Control over loop/prompt | Total | Limited | High |
| Tool execution | Your code | Lambda or `RETURN_CONTROL` | Your `@tool` fns |
| Multi-agent / custom orchestration | DIY | Limited | First-class (notebook 05) |
| Best for | Learning; bespoke control | Fast managed agent in-console | Production code-first agents |

**Skeptic's corner.** "If Strands writes the loop, why ever use Way 1?" — Because when something misbehaves (a tool never gets called, a retry storms, a prompt leaks), you debug it faster if you *know* the loop underneath. And some teams need control the framework doesn't expose. Way 1 is the mental model; Strands is the daily driver.

**Why this matters for AgentCore:** AgentCore Runtime is **framework-agnostic** — it can host any of these. But a code-first agent (Way 3) is the cleanest thing to package and ship, and it's what the rest of this series deploys.


## Next

**`02_agentcore_runtime.ipynb`** — take this exact `flight_agent` and put it in production on **AgentCore Runtime**: local test → `configure` → `launch` → `invoke`, multi-turn sessions, what happens under the hood (microVM, CodeBuild, ECR), the new CLI vs the notebook-friendly toolkit, and the errors that bite first.
